
# From Fraud Prediction to Business Action
### An Explainable, Behavior-Aware Risk-to-Action Framework for Online Transaction Management

**Primary dataset:** IEEE-CIS Fraud Detection (`train_transaction.csv` + `train_identity.csv`)
**Secondary/validation dataset:** PaySim (`PS_20174392719_1491204439457_log.csv`)

This notebook implements the locked experimental matrix **E0 → E7**:

| Exp | Name | Question |
|---|---|---|
| E0 | Conventional baselines | How hard is the task out-of-the-box? |
| E1 | Model comparison | Which model family is strongest on PR-AUC? |
| E2 | Behavioral enrichment | Does historical behavior improve risk prediction? |
| E3 | Imbalance handling | Does oversampling help minority detection? |
| E4 | Behavioral ablation | Which behavioral signal groups matter? |
| E5 | Explainability (SHAP) | Can predictions be explained at global/local level? |
| E6 | Risk-to-Action | Can probabilities become APPROVE/REVIEW/BLOCK policies? |
| E7 | Cost / operational analysis + PaySim validation | Does the framework help operationally, and does the concept transfer? |

**Design rules enforced throughout:**
- Temporal train/val/test split — **no random leakage** on the primary experiments.
- Every behavioral aggregate is computed using **strictly past transactions only** (`shift(1)` / expanding windows).
- Oversampling (if used) happens **after** the split, on TRAIN only.
- IEEE-CIS and PaySim are **not** forced into a shared feature vector — they're connected through a **common business layer** (Amount / Time / Type / Entity / Target), not literal column equivalence.
- No claim of "continuous learning from feedback" — neither dataset provides real post-decision outcomes. Cost analysis is explicitly a **sensitivity analysis under hypothetical scenarios**, not ground truth.


## 0. Setup & Configuration

In [ ]:

# ============================================================
# 0.1 Imports
# ============================================================
import os
import gc
import json
import warnings
import time
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef, precision_recall_curve,
    confusion_matrix
)

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 80)

# Optional libraries — degrade gracefully if not installed in the Kaggle image
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost not available — M3 will be skipped.")

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print("lightgbm not available — M4 will be skipped.")

try:
    from imblearn.over_sampling import RandomOverSampler, SMOTE
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False
    print("imbalanced-learn not available — pip install imbalanced-learn if E3 SMOTE cell fails.")

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print("shap not available — pip install shap if E5 fails.")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:

# ============================================================
# 0.2 Config — paths & run-size controls
# ============================================================
# Set SAMPLE_N = None to run on the FULL dataset (slower, most faithful).
# Set SAMPLE_N = 100_000 (or any int) for fast iteration while building the paper.
CONFIG = {
    "ieee_transaction_path": "/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv",
    "ieee_identity_path":    "/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv",
    "paysim_path":           "/kaggle/input/datasets/ealaxi/paysim1/PS_20174392719_1491204439457_log.csv",

    "SAMPLE_N_IEEE": None,      # e.g. 150_000 for a fast run, None = full ~590k rows
    "SAMPLE_N_PAYSIM": None,    # e.g. 300_000 for a fast run, None = full ~6.3M rows

    "train_frac": 0.60,
    "val_frac":   0.20,
    "test_frac":  0.20,

    "entity_col_ieee": "card1",         # primary behavioral entity key for IEEE-CIS
    "velocity_windows_ieee": [3600 * h for h in [1, 6, 24, 168]],   # TransactionDT is in seconds -> 1h,6h,24h,7d
    "velocity_windows_paysim": [1, 6, 24],  # step is in hours -> 1h, 6h, 24h windows

    "output_dir": "/kaggle/working/outputs",
}
os.makedirs(CONFIG["output_dir"], exist_ok=True)
print(json.dumps({k: v for k, v in CONFIG.items() if "path" not in k}, indent=2))


## 1. Data Loading

We load `train_transaction` + `train_identity` (left join on `TransactionID`) as the **only**
labelled IEEE-CIS data (the competition's `test_*` files have no `isFraud` and are not used for
supervised experiments — held out silently by Kaggle). PaySim is loaded separately as the
secondary validation dataset.

In [ ]:

# ============================================================
# 1.1 Load IEEE-CIS
# ============================================================
def load_ieee(config):
    t0 = time.time()
    trans_path = Path(config["ieee_transaction_path"])
    ident_path = Path(config["ieee_identity_path"])
    assert trans_path.exists(), f"Not found: {trans_path}"

    nrows = config["SAMPLE_N_IEEE"]
    df_trans = pd.read_csv(trans_path, nrows=nrows)

    if ident_path.exists():
        df_ident = pd.read_csv(ident_path)
        df = df_trans.merge(df_ident, on="TransactionID", how="left")
    else:
        print("WARNING: identity file not found — proceeding with transaction table only.")
        df = df_trans

    df = df.sort_values("TransactionDT").reset_index(drop=True)
    print(f"IEEE-CIS loaded: {df.shape[0]:,} rows x {df.shape[1]} cols in {time.time()-t0:.1f}s")
    return df

df_ieee = load_ieee(CONFIG)
df_ieee.head(3)


In [ ]:

# ============================================================
# 1.2 Load PaySim
# ============================================================
def load_paysim(config):
    t0 = time.time()
    path = Path(config["paysim_path"])
    assert path.exists(), f"Not found: {path}"
    nrows = config["SAMPLE_N_PAYSIM"]
    df = pd.read_csv(path, nrows=nrows)
    df = df.sort_values("step").reset_index(drop=True)
    print(f"PaySim loaded: {df.shape[0]:,} rows x {df.shape[1]} cols in {time.time()-t0:.1f}s")
    return df

df_paysim = load_paysim(CONFIG)
df_paysim.head(3)


## 2. Data Diagnostics

In [ ]:

def diagnose(df, name, target_col, time_col):
    print(f"===== {name} =====")
    print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} cols")
    if target_col in df.columns:
        fraud_rate = df[target_col].mean() * 100
        print(f"Fraud cases: {int(df[target_col].sum()):,} ({fraud_rate:.3f}%)")
    if time_col in df.columns:
        print(f"Unique '{time_col}' values: {df[time_col].nunique():,}  "
              f"(range: {df[time_col].min()} -> {df[time_col].max()})")
    miss = df.isna().mean().sort_values(ascending=False)
    top_missing = miss[miss > 0].head(10)
    if len(top_missing):
        print("Top missing columns:")
        print((top_missing * 100).round(1).astype(str) + "%")
    print()
    return {"rows": df.shape[0], "cols": df.shape[1],
            "fraud_rate": df[target_col].mean() if target_col in df.columns else None,
            "n_time_unique": df[time_col].nunique() if time_col in df.columns else None}

diag_ieee = diagnose(df_ieee, "IEEE-CIS", "isFraud", "TransactionDT")
diag_paysim = diagnose(df_paysim, "PaySim", "isFraud", "step")


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_ieee["isFraud"].value_counts(normalize=True).rename({0: "legit", 1: "fraud"}).plot(
    kind="bar", ax=axes[0], color=["#4C72B0", "#C44E52"])
axes[0].set_title("IEEE-CIS class balance")
df_paysim["isFraud"].value_counts(normalize=True).rename({0: "legit", 1: "fraud"}).plot(
    kind="bar", ax=axes[1], color=["#4C72B0", "#C44E52"])
axes[1].set_title("PaySim class balance")
plt.tight_layout()
plt.show()


## 3. Feature Grouping (IEEE-CIS)

We deliberately do **not** throw all 394 columns into one model. Three groups:

- **Group 1 — Core transaction**: amount, product, card, address, email-domain fields.
- **Group 2 — Existing behavioral aggregates**: the dataset's own `C*`, `D*`, `V*`, `M*` groups (already engineered by the competition organisers, treated as given transaction-context signal).
- **Group 3 — Our engineered behavioral features**: leakage-safe, entity-level historical aggregates we compute ourselves in Section 4 (velocity, recency, deviation).

In [ ]:

def build_feature_groups(df):
    cols = df.columns.tolist()
    core_candidates = [c for c in [
        "TransactionAmt", "ProductCD",
        "card1", "card2", "card3", "card4", "card5", "card6",
        "addr1", "addr2", "dist1", "dist2",
        "P_emaildomain", "R_emaildomain",
    ] if c in cols]

    group_c = [c for c in cols if c.startswith("C") and c[1:].isdigit()]
    group_d = [c for c in cols if c.startswith("D") and c[1:].isdigit()]
    group_v = [c for c in cols if c.startswith("V") and c[1:].isdigit()]
    group_m = [c for c in cols if c.startswith("M") and c[1:].isdigit()]
    id_cols = [c for c in cols if c.startswith("id_")]
    device_cols = [c for c in ["DeviceType", "DeviceInfo"] if c in cols]

    existing_behavioral = group_c + group_d + group_v + group_m

    groups = {
        "core": core_candidates,
        "existing_behavioral": existing_behavioral,
        "C": group_c, "D": group_d, "V": group_v, "M": group_m,
        "identity": id_cols + device_cols,
    }
    return groups

FEATURE_GROUPS = build_feature_groups(df_ieee)
for k, v in FEATURE_GROUPS.items():
    print(f"{k:20s}: {len(v):4d} columns")


## 4. Temporal Split

**Non-negotiable rule**: earlier transactions train, later transactions validate/test. No random
shuffling for the primary experiment. Split ratios are configurable (`train_frac` / `val_frac` /
`test_frac`), applied on the time-sorted index — i.e. a pure **time-based cut**, not a random
stratified split.

In [ ]:

def temporal_split(df, time_col, train_frac, val_frac, test_frac):
    assert abs(train_frac + val_frac + test_frac - 1.0) < 1e-6
    df_sorted = df.sort_values(time_col).reset_index(drop=True)
    n = len(df_sorted)
    n_train = int(n * train_frac)
    n_val = int(n * val_frac)

    train_df = df_sorted.iloc[:n_train].copy()
    val_df   = df_sorted.iloc[n_train:n_train + n_val].copy()
    test_df  = df_sorted.iloc[n_train + n_val:].copy()

    print(f"Train: {len(train_df):,} rows  [{train_df[time_col].min()} -> {train_df[time_col].max()}]  "
          f"fraud={train_df['isFraud'].mean()*100:.3f}%")
    print(f"Val:   {len(val_df):,} rows  [{val_df[time_col].min()} -> {val_df[time_col].max()}]  "
          f"fraud={val_df['isFraud'].mean()*100:.3f}%")
    print(f"Test:  {len(test_df):,} rows  [{test_df[time_col].min()} -> {test_df[time_col].max()}]  "
          f"fraud={test_df['isFraud'].mean()*100:.3f}%")
    return train_df, val_df, test_df

print("--- IEEE-CIS temporal split ---")
ieee_train_idx, ieee_val_idx, ieee_test_idx = temporal_split(
    df_ieee, "TransactionDT", CONFIG["train_frac"], CONFIG["val_frac"], CONFIG["test_frac"]
)


## 5. Leakage-Safe Behavioral Feature Engineering (Group 3)

Core principle enforced everywhere below:

$$Feature_t = f(X_1, X_2, \dots, X_{t-1}) \quad \text{never } f(X_1, \dots, X_{t+1})$$

We compute these **causally** on the *entire* time-sorted dataset (via `shift(1)` / expanding
windows grouped by entity) **before** splitting. This is safe — a causal feature for row *t* only
ever looks backward in time, regardless of which split that row eventually lands in — and it lets
every entity's history flow continuously across the train/val/test boundary, which is how a
production system would actually behave (yesterday's history doesn't reset at a synthetic split
line).

Engineered signals, generalizing the PaySim behavioral columns already present in the JSON
(`orig_prev_tx_count`, `orig_prev_amount_mean`, `orig_prev_amount_std`, `amount_vs_orig_mean`,
`dest_prev_tx_count`) to IEEE-CIS:

- **Historical amount statistics** — running mean/std of this entity's past transaction amounts.
- **Amount deviation / ratio** — how unusual is *this* amount relative to the entity's own history.
- **Velocity** — transaction counts in trailing windows (1h / 6h / 24h / 7d).
- **Recency** — seconds since the entity's previous transaction.
- **Behavioral deviation** — current vs. historical, expressed as z-scores / ratios rather than raw values.

Entity key: `card1` (a proxy for "card/customer" — the closest stable identifier available in
IEEE-CIS; documented as a limitation, not ground truth).

In [ ]:

def add_behavioral_features_ieee(df, entity_col, amount_col, time_col, windows_seconds, eps=1e-6):
    '''Leakage-safe, entity-level historical behavioral features.
    Must be called on a dataframe already sorted by time_col (ascending).
    '''
    df = df.sort_values(time_col).reset_index(drop=True)
    g = df.groupby(entity_col, sort=False)

    # --- previous-transaction count & running amount stats (shift(1) => strictly past) ---
    df["beh_prev_tx_count"] = g.cumcount()
    df["beh_prev_amount_mean"] = g[amount_col].apply(lambda s: s.shift(1).expanding().mean()).reset_index(level=0, drop=True)
    df["beh_prev_amount_std"]  = g[amount_col].apply(lambda s: s.shift(1).expanding().std()).reset_index(level=0, drop=True)
    df["beh_prev_amount_std"] = df["beh_prev_amount_std"].fillna(0.0)

    # --- amount deviation / ratio vs. own history ---
    df["beh_amount_zscore"] = (df[amount_col] - df["beh_prev_amount_mean"]) / (df["beh_prev_amount_std"] + eps)
    df["beh_amount_ratio"]  = df[amount_col] / (df["beh_prev_amount_mean"] + eps)
    # first-ever transaction for an entity has no history -> neutralize rather than leave NaN/inf
    first_tx_mask = df["beh_prev_tx_count"] == 0
    df.loc[first_tx_mask, ["beh_amount_zscore", "beh_amount_ratio"]] = 0.0
    df["beh_amount_ratio"] = df["beh_amount_ratio"].replace([np.inf, -np.inf], np.nan)
    df["beh_amount_ratio"] = df["beh_amount_ratio"].fillna(df["beh_amount_ratio"].median())

    # --- recency: seconds since this entity's previous transaction ---
    df["beh_time_since_prev"] = df[time_col] - g[time_col].shift(1)
    max_recency = df["beh_time_since_prev"].max()
    df["beh_time_since_prev"] = df["beh_time_since_prev"].fillna(max_recency if pd.notna(max_recency) else 0)

    # --- velocity: trailing-window transaction counts (strictly past, causal) ---
    # Implemented via a per-entity searchsorted scan for efficiency at scale.
    times = df[time_col].values
    entities = df[entity_col].fillna("__NA__").values
    order = np.argsort(entities, kind="stable")
    df_sorted_by_entity = df.iloc[order]
    ent_vals = df_sorted_by_entity[entity_col].fillna("__NA__").values
    t_vals = df_sorted_by_entity[time_col].values

    for w in windows_seconds:
        counts = np.zeros(len(df_sorted_by_entity), dtype=np.int32)
        start_idx = 0
        n = len(df_sorted_by_entity)
        i = 0
        while i < n:
            j = i
            while j < n and ent_vals[j] == ent_vals[i]:
                j += 1
            # rows i:j belong to one entity, already time-sorted (stable sort preserved order)
            sub_t = t_vals[i:j]
            lo_ptr = 0
            for k in range(len(sub_t)):
                cutoff = sub_t[k] - w
                while lo_ptr < k and sub_t[lo_ptr] < cutoff:
                    lo_ptr += 1
                counts[i + k] = k - lo_ptr  # transactions strictly before k, within window
            i = j
        colname = f"beh_velocity_{w}s"
        tmp = pd.Series(counts, index=df_sorted_by_entity.index, name=colname)
        df[colname] = tmp.reindex(df.index)

    return df

print("Behavioral feature function defined (windows in seconds):", CONFIG["velocity_windows_ieee"])


In [ ]:
t0 = time.time()
df_ieee_feat = add_behavioral_features_ieee(
    df_ieee,
    entity_col=CONFIG["entity_col_ieee"],
    amount_col="TransactionAmt",
    time_col="TransactionDT",
    windows_seconds=CONFIG["velocity_windows_ieee"],
)
print(f"Behavioral features added in {time.time()-t0:.1f}s. New shape: {df_ieee_feat.shape}")

GROUP3_COLS = [c for c in df_ieee_feat.columns if c.startswith("beh_")]
FEATURE_GROUPS["engineered_behavioral"] = GROUP3_COLS
print("Group 3 (engineered behavioral) columns:", GROUP3_COLS)

# Note: beh_prev_amount_mean/std are NaN for an entity's very first-ever transaction
# (no history exists yet) -- this is expected, not a bug. It is resolved downstream by
# SimplePreprocessor (Sec. 6), which median-imputes using statistics fit on TRAIN only.
print("\nRemaining NaNs (resolved later by the leakage-safe imputer, fit on TRAIN only):")
print(df_ieee_feat[GROUP3_COLS].isna().sum()[lambda s: s > 0])
df_ieee_feat[GROUP3_COLS].describe().T


### 5.1 Final temporal split (post feature-engineering)

Re-run the same time-based split, now on `df_ieee_feat` (which carries the causal behavioral
columns). Because the features were computed causally on the full timeline, this split remains
leakage-free.

In [ ]:

print("--- IEEE-CIS temporal split (final, with engineered features) ---")
ieee_train, ieee_val, ieee_test = temporal_split(
    df_ieee_feat, "TransactionDT", CONFIG["train_frac"], CONFIG["val_frac"], CONFIG["test_frac"]
)


### 5.2 Behavioral features for PaySim (secondary dataset)

Same causal logic, generalized. PaySim's entity key is `nameOrig`, its amount column is `amount`,
and its time column is `step` (already coarse — 1 unit = 1 simulated hour), so the velocity
windows are expressed in **hours**, not seconds (see `CONFIG["velocity_windows_paysim"]`).

In [ ]:

def add_behavioral_features_paysim(df, windows_hours, eps=1e-6):
    df = df.sort_values("step").reset_index(drop=True)
    g = df.groupby("nameOrig", sort=False)

    df["beh_prev_tx_count"] = g.cumcount()
    df["beh_prev_amount_mean"] = g["amount"].apply(lambda s: s.shift(1).expanding().mean()).reset_index(level=0, drop=True)
    df["beh_prev_amount_std"]  = g["amount"].apply(lambda s: s.shift(1).expanding().std()).reset_index(level=0, drop=True).fillna(0.0)
    df["beh_amount_zscore"] = (df["amount"] - df["beh_prev_amount_mean"]) / (df["beh_prev_amount_std"] + eps)
    df["beh_amount_ratio"]  = df["amount"] / (df["beh_prev_amount_mean"] + eps)
    first_tx_mask = df["beh_prev_tx_count"] == 0
    df.loc[first_tx_mask, ["beh_amount_zscore", "beh_amount_ratio"]] = 0.0
    df["beh_amount_ratio"] = df["beh_amount_ratio"].replace([np.inf, -np.inf], np.nan)
    df["beh_amount_ratio"] = df["beh_amount_ratio"].fillna(df["beh_amount_ratio"].median())

    df["dest_prev_tx_count"] = df.groupby("nameDest", sort=False).cumcount()

    for w in windows_hours:
        colname = f"beh_velocity_{w}h"
        df[colname] = df.groupby("nameOrig", sort=False)["step"].transform(
            lambda s: s.apply(lambda t: ((s < t) & (s >= t - w)).sum())
        )
    return df

print("PaySim behavioral feature function defined (windows in hours):", CONFIG["velocity_windows_paysim"])


## 6. Preprocessing Pipeline

Categorical encoding and imputation are **fit on TRAIN only** and applied to val/test, to avoid
statistical leakage of distributional information from the future. We use:

- Frequency encoding for high-cardinality categoricals (`card1`-`card6`, `addr1/2`, email domains, `DeviceInfo`, id_* strings).
- Median imputation for numeric columns.
- A boolean "was-missing" flag for columns with heavy missingness (IEEE-CIS is known to have some columns like `D7` with ~90%+ missing).

In [ ]:

def get_dtype_split(df, feature_cols):
    cat_cols = [c for c in feature_cols if df[c].dtype == "object" or str(df[c].dtype) == "category"]
    num_cols = [c for c in feature_cols if c not in cat_cols]
    return cat_cols, num_cols


class SimplePreprocessor:
    '''Fit on TRAIN, transform TRAIN/VAL/TEST identically. Frequency-encodes categoricals,
    median-imputes numerics, adds missingness flags for very sparse columns.'''

    def __init__(self, feature_cols, missing_flag_threshold=0.5):
        self.feature_cols = feature_cols
        self.missing_flag_threshold = missing_flag_threshold
        self.freq_maps = {}
        self.medians = {}
        self.flag_cols = []
        self.cat_cols = []
        self.num_cols = []

    def fit(self, df_train):
        self.cat_cols, self.num_cols = get_dtype_split(df_train, self.feature_cols)
        miss_rate = df_train[self.feature_cols].isna().mean()
        self.flag_cols = miss_rate[miss_rate > self.missing_flag_threshold].index.tolist()

        for c in self.cat_cols:
            vc = df_train[c].astype(str).value_counts(normalize=True)
            self.freq_maps[c] = vc.to_dict()
        for c in self.num_cols:
            self.medians[c] = df_train[c].median()
        return self

    def transform(self, df):
        out = pd.DataFrame(index=df.index)
        for c in self.flag_cols:
            out[f"{c}_was_missing"] = df[c].isna().astype(np.int8)
        for c in self.cat_cols:
            freq = self.freq_maps.get(c, {})
            out[c] = df[c].astype(str).map(freq).fillna(0.0)
        for c in self.num_cols:
            med = self.medians.get(c, 0.0)
            out[c] = df[c].fillna(med)
        return out

    def fit_transform(self, df_train):
        return self.fit(df_train).transform(df_train)


def make_xy(train_df, val_df, test_df, feature_cols, target_col="isFraud", scale=False):
    prep = SimplePreprocessor(feature_cols).fit(train_df)
    X_train = prep.transform(train_df)
    X_val   = prep.transform(val_df)
    X_test  = prep.transform(test_df)
    y_train, y_val, y_test = train_df[target_col].values, val_df[target_col].values, test_df[target_col].values

    if scale:
        scaler = StandardScaler()
        X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
        X_val   = pd.DataFrame(scaler.transform(X_val),   columns=X_val.columns,   index=X_val.index)
        X_test  = pd.DataFrame(scaler.transform(X_test),  columns=X_test.columns,  index=X_test.index)

    return X_train, y_train, X_val, y_val, X_test, y_test, prep


## 7. Evaluation Metrics

**Primary**: PR-AUC (average precision) — the right primary metric under heavy class imbalance.
Secondary: ROC-AUC, Precision, Recall, F1, MCC at the default 0.5 threshold (for reference only —
Section 12/E6 replaces the single threshold with a cost-optimized policy).

In [ ]:

def evaluate(y_true, y_proba, threshold=0.5, label=""):
    y_pred = (y_proba >= threshold).astype(int)
    metrics = {
        "label": label,
        "PR_AUC": average_precision_score(y_true, y_proba),
        "ROC_AUC": roc_auc_score(y_true, y_proba),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred),
    }
    return metrics

results_log = []  # accumulates dict rows across every experiment in this notebook


## 8. E0 — Conventional Baselines & E1 — Model Comparison

Four model families, all using **class-weighting** (not oversampling — that's tested separately
in E3) as the primary imbalance strategy:

- **M1** Logistic Regression (interpretability baseline, needs scaled features)
- **M2** Random Forest (nonlinear bagging baseline)
- **M3** XGBoost (gradient boosting)
- **M4** LightGBM (efficient gradient boosting)

Feature set used here: **core + existing_behavioral** (Groups 1+2) — the "conventional" feature
set, before we test whether our own engineered behavioral features (Group 3) add anything in E2.

In [ ]:

BASELINE_FEATURES = FEATURE_GROUPS["core"] + FEATURE_GROUPS["existing_behavioral"]
print(f"E0/E1 baseline feature set: {len(BASELINE_FEATURES)} columns")

X_train, y_train, X_val, y_val, X_test, y_test, prep_baseline = make_xy(
    ieee_train, ieee_val, ieee_test, BASELINE_FEATURES, scale=False
)
X_train_scaled, _, X_val_scaled, _, X_test_scaled, _, _ = make_xy(
    ieee_train, ieee_val, ieee_test, BASELINE_FEATURES, scale=True
)
print(X_train.shape, X_val.shape, X_test.shape)


In [ ]:

def get_model_zoo(pos_weight):
    models = {}
    models["M1_LogReg"] = ("scaled", LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
    ))
    models["M2_RandomForest"] = ("raw", RandomForestClassifier(
        n_estimators=300, max_depth=12, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    ))
    if HAS_XGB:
        models["M3_XGBoost"] = ("raw", xgb.XGBClassifier(
            n_estimators=400, max_depth=6, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=pos_weight, eval_metric="aucpr",
            random_state=RANDOM_STATE, n_jobs=-1, tree_method="hist"
        ))
    if HAS_LGB:
        models["M4_LightGBM"] = ("raw", lgb.LGBMClassifier(
            n_estimators=400, max_depth=-1, num_leaves=63, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=pos_weight, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
        ))
    return models


def run_model_zoo(X_train, y_train, X_val, y_val, X_test, y_test,
                   X_train_scaled, X_val_scaled, X_test_scaled, tag=""):
    pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
    models = get_model_zoo(pos_weight)
    fitted = {}
    rows = []
    for name, (kind, model) in models.items():
        Xtr, Xva, Xte = (X_train_scaled, X_val_scaled, X_test_scaled) if kind == "scaled" else (X_train, X_val, X_test)
        t0 = time.time()
        model.fit(Xtr, y_train)
        fit_time = time.time() - t0
        proba_val = model.predict_proba(Xva)[:, 1]
        proba_test = model.predict_proba(Xte)[:, 1]
        m_val = evaluate(y_val, proba_val, label=f"{tag}{name}_VAL")
        m_test = evaluate(y_test, proba_test, label=f"{tag}{name}_TEST")
        m_val["fit_time_s"] = m_test["fit_time_s"] = round(fit_time, 1)
        rows.append(m_val); rows.append(m_test)
        fitted[name] = model
        print(f"{name:16s} fit={fit_time:6.1f}s  VAL PR-AUC={m_val['PR_AUC']:.4f}  TEST PR-AUC={m_test['PR_AUC']:.4f}")
    return fitted, pd.DataFrame(rows)

print("--- E0/E1: baseline model comparison (core + existing behavioral features) ---")
fitted_baseline, df_e1 = run_model_zoo(
    X_train, y_train, X_val, y_val, X_test, y_test,
    X_train_scaled, X_val_scaled, X_test_scaled, tag="E1_"
)
results_log.append(df_e1)
df_e1.sort_values("PR_AUC", ascending=False)


In [ ]:

best_model_name = (
    df_e1[df_e1.label.str.contains("_VAL")]
    .assign(base=lambda d: d.label.str.replace("E1_", "").str.replace("_VAL", ""))
    .sort_values("PR_AUC", ascending=False)
    .iloc[0]["base"]
)
print("Best model on VAL PR-AUC:", best_model_name)


## 9. E2 — Behavioral Enrichment

Three progressively richer feature sets, all trained with the **same** model type (the E1 winner)
so any PR-AUC delta is attributable to the features, not the algorithm:

- **Model A** — Core only (Group 1)
- **Model B** — Core + existing dataset behavioral aggregates (Groups 1+2) — same as the E1 baseline set
- **Model C** — Core + existing + our engineered historical behavioral features (Groups 1+2+3)

We report $\Delta$PR-AUC and $\Delta$MCC (C vs. B) as the headline evidence for "does behavioral
history help".

In [ ]:

def fit_single_model(name, X_train, y_train, X_val, y_val, X_test, y_test, X_train_scaled=None, X_val_scaled=None, X_test_scaled=None):
    pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
    zoo = get_model_zoo(pos_weight)
    kind, model = zoo[name]
    if kind == "scaled":
        model.fit(X_train_scaled, y_train)
        proba_val, proba_test = model.predict_proba(X_val_scaled)[:, 1], model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        proba_val, proba_test = model.predict_proba(X_val)[:, 1], model.predict_proba(X_test)[:, 1]
    return model, proba_val, proba_test


feature_sets_e2 = {
    "ModelA_core_only": FEATURE_GROUPS["core"],
    "ModelB_core_plus_existing": FEATURE_GROUPS["core"] + FEATURE_GROUPS["existing_behavioral"],
    "ModelC_core_existing_engineered": (
        FEATURE_GROUPS["core"] + FEATURE_GROUPS["existing_behavioral"] + FEATURE_GROUPS["engineered_behavioral"]
    ),
}

e2_rows = []
e2_models = {}
for fs_name, cols in feature_sets_e2.items():
    cols = [c for c in cols if c in ieee_train.columns]
    scale_needed = best_model_name == "M1_LogReg"
    Xtr, ytr, Xva, yva, Xte, yte, _ = make_xy(ieee_train, ieee_val, ieee_test, cols, scale=scale_needed)
    if scale_needed:
        model, pv, pt = fit_single_model(best_model_name, Xtr, ytr, Xva, yva, Xte, yte, Xtr, Xva, Xte)
    else:
        model, pv, pt = fit_single_model(best_model_name, Xtr, ytr, Xva, yva, Xte, yte)
    m_val = evaluate(yva, pv, label=f"E2_{fs_name}_VAL")
    m_test = evaluate(yte, pt, label=f"E2_{fs_name}_TEST")
    e2_rows += [m_val, m_test]
    e2_models[fs_name] = model
    print(f"{fs_name:35s} n_feat={len(cols):4d}  VAL PR-AUC={m_val['PR_AUC']:.4f}  TEST PR-AUC={m_test['PR_AUC']:.4f}")

df_e2 = pd.DataFrame(e2_rows)
results_log.append(df_e2)
df_e2


In [ ]:

test_rows = df_e2[df_e2.label.str.contains("_TEST")].copy()
test_rows["feature_set"] = test_rows.label.str.replace("E2_", "").str.replace("_TEST", "")
pr_b = test_rows.loc[test_rows.feature_set == "ModelB_core_plus_existing", "PR_AUC"].values[0]
pr_c = test_rows.loc[test_rows.feature_set == "ModelC_core_existing_engineered", "PR_AUC"].values[0]
mcc_b = test_rows.loc[test_rows.feature_set == "ModelB_core_plus_existing", "MCC"].values[0]
mcc_c = test_rows.loc[test_rows.feature_set == "ModelC_core_existing_engineered", "MCC"].values[0]
print(f"Delta PR-AUC (C vs B): {pr_c - pr_b:+.4f}")
print(f"Delta MCC   (C vs B): {mcc_c - mcc_b:+.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=test_rows, x="feature_set", y="PR_AUC", ax=ax, palette="viridis")
ax.set_title("E2 — Behavioral enrichment (TEST PR-AUC)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout(); plt.show()


## 10. E3 — Imbalance Handling

Uses the **best feature set from E2** (Model C). Rule enforced: split first, oversample TRAIN
only, VAL/TEST untouched.

- **E3.0** — No oversampling (class-weighting only, same as above)
- **E3.1** — Random oversampling of the minority class
- **E3.2** — SMOTE on a clean numerical, model-ready representation (not naive SMOTE across all
  394 raw/mixed-type IEEE-CIS columns — see Section 13 of the plan)

In [ ]:

best_feature_set = feature_sets_e2["ModelC_core_existing_engineered"]
best_feature_set = [c for c in best_feature_set if c in ieee_train.columns]
scale_needed = best_model_name == "M1_LogReg"
Xtr, ytr, Xva, yva, Xte, yte, prep_e3 = make_xy(ieee_train, ieee_val, ieee_test, best_feature_set, scale=scale_needed)

e3_rows = []
e3_results = {}

# E3.0 — no oversampling (reuse E2 Model C fit)
model_e30, pv, pt = fit_single_model(best_model_name, Xtr, ytr, Xva, yva, Xte, yte, Xtr, Xva, Xte)
e3_rows.append(evaluate(yva, pv, label="E3_none_VAL"))
e3_rows.append(evaluate(yte, pt, label="E3_none_TEST"))
e3_results["none"] = model_e30

if HAS_IMBLEARN:
    # E3.1 — random oversampling (TRAIN only)
    ros = RandomOverSampler(random_state=RANDOM_STATE)
    Xtr_ros, ytr_ros = ros.fit_resample(Xtr, ytr)
    model_e31, pv, pt = fit_single_model(best_model_name, Xtr_ros, ytr_ros, Xva, yva, Xte, yte, Xtr_ros, Xva, Xte)
    e3_rows.append(evaluate(yva, pv, label="E3_random_oversample_VAL"))
    e3_rows.append(evaluate(yte, pt, label="E3_random_oversample_TEST"))
    e3_results["random_oversample"] = model_e31

    # E3.2 — SMOTE (TRAIN only, on the already-numeric preprocessed matrix)
    smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
    Xtr_smote, ytr_smote = smote.fit_resample(Xtr, ytr)
    model_e32, pv, pt = fit_single_model(best_model_name, Xtr_smote, ytr_smote, Xva, yva, Xte, yte, Xtr_smote, Xva, Xte)
    e3_rows.append(evaluate(yva, pv, label="E3_smote_VAL"))
    e3_rows.append(evaluate(yte, pt, label="E3_smote_TEST"))
    e3_results["smote"] = model_e32
else:
    print("imbalanced-learn not installed — run `!pip install imbalanced-learn` and re-run this cell for E3.1/E3.2.")

df_e3 = pd.DataFrame(e3_rows)
results_log.append(df_e3)
df_e3


## 11. E4 — Behavioral Ablation

Starting from Model C (full feature set), remove one behavioral component at a time:

- **A1** — without velocity features
- **A2** — without historical amount statistics (mean/std)
- **A3** — without deviation features (z-score/ratio)
- **A4** — without temporal/recency feature
- **A5** — full model (reference)

In [ ]:

velocity_cols = [c for c in GROUP3_COLS if "velocity" in c]
amount_stat_cols = [c for c in GROUP3_COLS if c in ("beh_prev_amount_mean", "beh_prev_amount_std")]
deviation_cols = [c for c in GROUP3_COLS if c in ("beh_amount_zscore", "beh_amount_ratio")]
recency_cols = [c for c in GROUP3_COLS if c == "beh_time_since_prev"]

base_cols = FEATURE_GROUPS["core"] + FEATURE_GROUPS["existing_behavioral"]

ablations = {
    "A1_without_velocity":        base_cols + [c for c in GROUP3_COLS if c not in velocity_cols],
    "A2_without_amount_stats":    base_cols + [c for c in GROUP3_COLS if c not in amount_stat_cols],
    "A3_without_deviation":       base_cols + [c for c in GROUP3_COLS if c not in deviation_cols],
    "A4_without_recency":         base_cols + [c for c in GROUP3_COLS if c not in recency_cols],
    "A5_full_model":              base_cols + GROUP3_COLS,
}

e4_rows = []
for abl_name, cols in ablations.items():
    cols = [c for c in cols if c in ieee_train.columns]
    scale_needed = best_model_name == "M1_LogReg"
    Xtr, ytr, Xva, yva, Xte, yte, _ = make_xy(ieee_train, ieee_val, ieee_test, cols, scale=scale_needed)
    model, pv, pt = fit_single_model(best_model_name, Xtr, ytr, Xva, yva, Xte, yte, Xtr, Xva, Xte)
    m_test = evaluate(yte, pt, label=f"E4_{abl_name}_TEST")
    e4_rows.append(m_test)
    print(f"{abl_name:28s} n_feat={len(cols):4d}  TEST PR-AUC={m_test['PR_AUC']:.4f}  MCC={m_test['MCC']:.4f}")

df_e4 = pd.DataFrame(e4_rows)
results_log.append(df_e4)

fig, ax = plt.subplots(figsize=(8, 4))
plot_df = df_e4.copy()
plot_df["ablation"] = plot_df.label.str.replace("E4_", "").str.replace("_TEST", "")
sns.barplot(data=plot_df, x="ablation", y="PR_AUC", ax=ax, palette="mako")
ax.set_title("E4 — Behavioral ablation (TEST PR-AUC)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout(); plt.show()
df_e4


## 12. E5 — Explainability (SHAP)

Single XAI method (SHAP), used at three levels on the **final model** (Model C / A5, the full
behavioral feature set):

- **Global** — which features generally drive fraud risk (mean |SHAP|).
- **Class-level** — SHAP summary/beeswarm distinguishing fraud vs. legitimate.
- **Local** — a waterfall-style explanation for one specific high-risk transaction, which we then
  translate into the "KEY RISK DRIVERS" business narrative used in Section 13.

In [ ]:

final_cols = [c for c in (FEATURE_GROUPS["core"] + FEATURE_GROUPS["existing_behavioral"] + GROUP3_COLS) if c in ieee_train.columns]
scale_needed = best_model_name == "M1_LogReg"
X_train_final, y_train_final, X_val_final, y_val_final, X_test_final, y_test_final, prep_final = make_xy(
    ieee_train, ieee_val, ieee_test, final_cols, scale=scale_needed
)
final_model, proba_val_final, proba_test_final = fit_single_model(
    best_model_name, X_train_final, y_train_final, X_val_final, y_val_final, X_test_final, y_test_final,
    X_train_final, X_val_final, X_test_final
)
final_metrics_test = evaluate(y_test_final, proba_test_final, label="FINAL_MODEL_TEST")
print(final_metrics_test)


In [ ]:

if HAS_SHAP:
    # Use a manageable background/explain sample for speed on tree models
    explain_sample = X_test_final.sample(n=min(2000, len(X_test_final)), random_state=RANDOM_STATE)

    if best_model_name in ("M2_RandomForest", "M3_XGBoost", "M4_LightGBM"):
        explainer = shap.TreeExplainer(final_model)
        shap_values = explainer.shap_values(explain_sample)
        if isinstance(shap_values, list):  # some sklearn RF versions return [class0, class1]
            shap_values = shap_values[1]
    else:
        background = X_train_final.sample(n=min(200, len(X_train_final)), random_state=RANDOM_STATE)
        explainer = shap.LinearExplainer(final_model, background)
        shap_values = explainer.shap_values(explain_sample)

    print("Global feature importance (mean |SHAP|):")
    shap.summary_plot(shap_values, explain_sample, plot_type="bar", show=True)
else:
    print("Install shap (`pip install shap`) to run this cell.")


In [ ]:

if HAS_SHAP:
    print("Class-level explanation (beeswarm):")
    shap.summary_plot(shap_values, explain_sample, show=True)


In [ ]:

def explain_transaction_as_business_narrative(shap_row, feature_names, proba, top_k=3, action_thresholds=(0.3, 0.7)):
    '''Turn a raw SHAP row into the 'RISK SCORE / KEY RISK DRIVERS / RECOMMENDED ACTION' narrative.'''
    order = np.argsort(-np.abs(shap_row))[:top_k]
    drivers = []
    for idx in order:
        direction = "↑" if shap_row[idx] > 0 else "↓"
        drivers.append(f"{direction} {feature_names[idx]} (SHAP={shap_row[idx]:+.3f})")

    tau1, tau2 = action_thresholds
    action = "APPROVE" if proba < tau1 else ("REVIEW" if proba < tau2 else "BLOCK")

    print("RISK SCORE")
    print(f"{proba:.2f}\n")
    print("KEY RISK DRIVERS")
    for d in drivers:
        print(d)
    print(f"\nRECOMMENDED ACTION\n{action}")

if HAS_SHAP:
    # pick the highest-probability transaction in the explain sample as the local example
    local_idx = int(np.argmax(final_model.predict_proba(explain_sample)[:, 1]))
    local_proba = final_model.predict_proba(explain_sample)[local_idx, 1]
    explain_transaction_as_business_narrative(
        shap_values[local_idx], explain_sample.columns.tolist(), local_proba
    )


## 13. E6 — Risk-to-Action: Cost-Sensitive Thresholds

We replace the naive $p>0.5 \Rightarrow$ Fraud rule with a **three-way policy**:

$$
A(p)=\begin{cases}
\text{APPROVE} & p<\tau_1\\
\text{REVIEW} & \tau_1\le p<\tau_2\\
\text{BLOCK} & p\ge\tau_2
\end{cases}
$$

**Important caveat (kept explicit, as required):** neither dataset contains real bank-specific
financial costs or true post-decision outcomes. What follows is a **cost-sensitivity analysis
under explicitly hypothetical, normalized cost scenarios** — not a ground-truth cost estimate.

In [ ]:

def expected_cost(y_true, proba, tau1, tau2, c_fn, c_fp, c_review, amounts=None):
    '''
    APPROVE region (p < tau1): missed fraud costs c_fn * (fraud amount or unit cost); legit costs 0.
    REVIEW region (tau1 <= p < tau2): every case costs c_review (manual investigation), regardless of true label.
    BLOCK region (p >= tau2): legit-but-blocked costs c_fp (false decline); true fraud blocked costs 0 (prevented).
    If `amounts` is provided, c_fn is scaled per-transaction by amount; otherwise a flat unit cost is used.
    '''
    action = np.where(proba < tau1, "APPROVE", np.where(proba < tau2, "REVIEW", "BLOCK"))
    cost = np.zeros(len(y_true), dtype=float)

    fn_mask = (action == "APPROVE") & (y_true == 1)
    fp_mask = (action == "BLOCK") & (y_true == 0)
    rv_mask = (action == "REVIEW")

    if amounts is not None:
        cost[fn_mask] = c_fn * amounts[fn_mask]
    else:
        cost[fn_mask] = c_fn
    cost[fp_mask] = c_fp
    cost[rv_mask] = c_review

    total = cost.sum()
    n_review = rv_mask.sum()
    fraud_capture = y_true[action != "APPROVE"].sum() / max(y_true.sum(), 1)
    review_rate = n_review / len(y_true)
    false_decline_rate = fp_mask.sum() / max((y_true == 0).sum(), 1)
    return {
        "tau1": tau1, "tau2": tau2, "total_cost": total,
        "fraud_capture": fraud_capture, "review_rate": review_rate,
        "false_decline_rate": false_decline_rate, "n_review": int(n_review),
    }


def optimize_thresholds(y_true, proba, c_fn, c_fp, c_review, amounts=None, grid=41):
    taus = np.linspace(0.01, 0.99, grid)
    best = None
    rows = []
    for tau1 in taus:
        for tau2 in taus:
            if tau2 <= tau1:
                continue
            r = expected_cost(y_true, proba, tau1, tau2, c_fn, c_fp, c_review, amounts)
            rows.append(r)
            if best is None or r["total_cost"] < best["total_cost"]:
                best = r
    return best, pd.DataFrame(rows)


In [ ]:

amounts_test = ieee_test.loc[:, "TransactionAmt"].values if "TransactionAmt" in ieee_test.columns else None

# Normalized hypothetical cost scenarios: fraud loss expressed as a multiple of a flat false-decline cost.
# c_review is kept small relative to c_fp, reflecting that manual review is cheaper than either extreme.
scenarios = {
    "S1_fraud_10x": dict(c_fn=10.0, c_fp=1.0, c_review=0.2),
    "S2_fraud_25x": dict(c_fn=25.0, c_fp=1.0, c_review=0.2),
    "S3_fraud_50x": dict(c_fn=50.0, c_fp=1.0, c_review=0.2),
}

e6_rows = []
policy1_rows = []  # conventional p>0.5 => BLOCK, for comparison
for scen_name, costs in scenarios.items():
    best, grid_df = optimize_thresholds(y_test_final, proba_test_final, amounts=None, **costs)
    best["scenario"] = scen_name
    e6_rows.append(best)
    print(f"{scen_name}: tau1*={best['tau1']:.3f}  tau2*={best['tau2']:.3f}  "
          f"cost={best['total_cost']:.1f}  fraud_capture={best['fraud_capture']:.3f}  "
          f"review_rate={best['review_rate']:.3f}  FDR={best['false_decline_rate']:.4f}")

    conv = expected_cost(y_test_final, proba_test_final, tau1=0.5, tau2=0.5 + 1e-9, **costs)
    conv["scenario"] = scen_name
    policy1_rows.append(conv)

df_e6 = pd.DataFrame(e6_rows)
df_policy1 = pd.DataFrame(policy1_rows)
results_log.append(df_e6)
print("\nConventional p>0.5 policy, for comparison:")
df_policy1


In [ ]:

fig, ax = plt.subplots(figsize=(7, 4))
compare = pd.concat([
    df_e6.assign(policy="Cost-optimized (Policy 3)"),
    df_policy1.assign(policy="Conventional p>0.5 (Policy 1)"),
])
sns.barplot(data=compare, x="scenario", y="total_cost", hue="policy", ax=ax)
ax.set_title("Policy 3 (cost-optimized) vs. Policy 1 (conventional) — total hypothetical cost")
plt.tight_layout(); plt.show()


## 14. E7a — Operational Metrics (Review-Budget Analysis)

Beyond a single F1 number, we report metrics an operations team actually cares about:

- **Fraud Capture** — share of all fraud caught by REVIEW+BLOCK.
- **Review Rate** — share of traffic sent to manual review.
- **False Decline Rate** — share of legitimate transactions wrongly blocked.
- **Fraud Capture @ Review Budget** — "if analysts can only look at X% of traffic, how much fraud
  do we prioritize into their queue?" (FC@1%, FC@5%, FC@10%).

In [ ]:

def fraud_capture_at_budget(y_true, proba, budget_fracs=(0.01, 0.05, 0.10)):
    order = np.argsort(-proba)  # highest risk first
    y_sorted = y_true[order]
    n = len(y_true)
    total_fraud = y_true.sum()
    out = {}
    for b in budget_fracs:
        k = max(int(np.ceil(n * b)), 1)
        captured = y_sorted[:k].sum()
        out[f"FC@{int(b*100)}%"] = captured / max(total_fraud, 1)
    return out

fc_budget = fraud_capture_at_budget(y_test_final, proba_test_final)
print("Fraud Capture @ Review Budget (final model, TEST):")
for k, v in fc_budget.items():
    print(f"  {k}: {v:.3f}")

precision, recall, thresh = precision_recall_curve(y_test_final, proba_test_final)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(recall, precision)
ax.set_xlabel("Recall (Fraud Capture)"); ax.set_ylabel("Precision")
ax.set_title(f"Precision-Recall curve — final model (PR-AUC={final_metrics_test['PR_AUC']:.3f})")
plt.tight_layout(); plt.show()


## 15. E7b — Cross-Dataset Conceptual Validation (PaySim)

**Important**: IEEE-CIS and PaySim have different schemas. We do **not** train on one and test on
the literal same feature vector on the other. Instead we validate the *conceptual framework*
(Behavior → Risk → Explanation → Action) by running the same pipeline shape on PaySim, connected
through a common **business layer**:

| Business concept   | IEEE-CIS       | PaySim   |
|---------------------|----------------|----------|
| Amount               | `TransactionAmt` | `amount`   |
| Time                  | `TransactionDT`  | `step`   |
| Transaction type      | `ProductCD`      | `type`   |
| Entity                | `card1`          | `nameOrig` |
| Target                 | `isFraud`        | `isFraud` |
| Behavioral history     | derived (Sec. 5) | derived (Sec. 5.2) |

**Note on PaySim's balance columns**: `oldbalanceOrg`, `newbalanceOrig`, `oldbalanceDest`,
`newbalanceDest` have documented simulation artifacts in PaySim (e.g. zero-balance patterns
correlated with fraud by construction of the simulator, not by real-world causal mechanism). We
keep them out of the "safe" feature set by default and treat them as a separate diagnostic, not a
feature to blindly trust.

In [ ]:

df_paysim_feat = add_behavioral_features_paysim(df_paysim, CONFIG["velocity_windows_paysim"])

print("--- PaySim temporal split ---")
paysim_train, paysim_val, paysim_test = temporal_split(
    df_paysim_feat, "step", CONFIG["train_frac"], CONFIG["val_frac"], CONFIG["test_frac"]
)


In [ ]:

PAYSIM_SAFE_CORE = ["amount", "type"]
PAYSIM_RAW_BALANCE_COLS = ["oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest"]  # flagged, not blindly trusted
PAYSIM_BEHAVIORAL = [c for c in df_paysim_feat.columns if c.startswith("beh_") or c == "dest_prev_tx_count"]

paysim_feature_cols = PAYSIM_SAFE_CORE + PAYSIM_BEHAVIORAL
paysim_feature_cols = [c for c in paysim_feature_cols if c in paysim_train.columns]
print("PaySim feature set (balance columns intentionally excluded from the safe baseline):")
print(paysim_feature_cols)

Xtr_ps, ytr_ps, Xva_ps, yva_ps, Xte_ps, yte_ps, prep_ps = make_xy(
    paysim_train, paysim_val, paysim_test, paysim_feature_cols, target_col="isFraud", scale=False
)

pos_weight_ps = (ytr_ps == 0).sum() / max((ytr_ps == 1).sum(), 1)
if HAS_LGB:
    paysim_model = lgb.LGBMClassifier(
        n_estimators=300, num_leaves=63, learning_rate=0.05,
        scale_pos_weight=pos_weight_ps, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
    )
elif HAS_XGB:
    paysim_model = xgb.XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        scale_pos_weight=pos_weight_ps, eval_metric="aucpr", random_state=RANDOM_STATE, n_jobs=-1
    )
else:
    paysim_model = RandomForestClassifier(
        n_estimators=300, max_depth=12, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
    )

paysim_model.fit(Xtr_ps, ytr_ps)
proba_test_ps = paysim_model.predict_proba(Xte_ps)[:, 1]
paysim_metrics = evaluate(yte_ps, proba_test_ps, label="PaySim_secondary_validation_TEST")
print(paysim_metrics)
results_log.append(pd.DataFrame([paysim_metrics]))


In [ ]:

fc_budget_ps = fraud_capture_at_budget(yte_ps, proba_test_ps)
print("PaySim — Fraud Capture @ Review Budget:")
for k, v in fc_budget_ps.items():
    print(f"  {k}: {v:.3f}")

print("\nConceptual comparison — does the same operational shape hold across both datasets?")
compare_df = pd.DataFrame({
    "IEEE-CIS (primary)": {**{k: v for k, v in final_metrics_test.items() if k not in ("label",)}, **fc_budget},
    "PaySim (secondary)": {**{k: v for k, v in paysim_metrics.items() if k not in ("label",)}, **fc_budget_ps},
}).T
compare_df


## 16. Save Artifacts

In [ ]:

out_dir = CONFIG["output_dir"]

all_results = pd.concat(results_log, ignore_index=True, sort=False)
all_results.to_csv(f"{out_dir}/all_experiment_results.csv", index=False)

for name, df_ in [("E1_model_comparison", df_e1), ("E2_behavioral_enrichment", df_e2),
                   ("E3_imbalance_handling", df_e3), ("E4_behavioral_ablation", df_e4),
                   ("E6_cost_optimization", df_e6)]:
    df_.to_csv(f"{out_dir}/{name}.csv", index=False)

compare_df.to_csv(f"{out_dir}/E7_ieee_vs_paysim_comparison.csv")

try:
    import joblib
    joblib.dump(final_model, f"{out_dir}/final_model_ieee_cis.pkl")
    joblib.dump(paysim_model, f"{out_dir}/secondary_model_paysim.pkl")
    print("Models saved.")
except ImportError:
    print("joblib not available — skip model persistence, results CSVs are still saved.")

print(f"\nAll artifacts written to: {out_dir}")
print(os.listdir(out_dir))


## 17. Summary

This notebook produced, in order:

1. **E0/E1** — a fair comparison of Logistic Regression / Random Forest / XGBoost / LightGBM under
   class-weighting on the conventional (core + existing-behavioral) feature set.
2. **E2** — quantified evidence (ΔPR-AUC, ΔMCC) for whether our leakage-safe, entity-level
   historical behavioral features (velocity, recency, deviation, running amount stats) improve
   fraud risk prediction over the conventional feature set.
3. **E3** — tested oversampling (random, SMOTE) against class-weighting, strictly on TRAIN, with
   VAL/TEST held untouched.
4. **E4** — an ablation isolating which behavioral signal groups actually drive performance.
5. **E5** — SHAP explanations at global, class, and single-transaction level, translated into a
   business-facing "RISK SCORE / KEY RISK DRIVERS / RECOMMENDED ACTION" narrative.
6. **E6** — replaced the naive `p > 0.5` rule with a cost-optimized three-way
   APPROVE/REVIEW/BLOCK policy, under three explicitly hypothetical cost scenarios (not claimed as
   ground truth).
7. **E7** — operational metrics (fraud capture at review budget, review rate, false decline rate)
   and a conceptual (not feature-literal) validation of the same pipeline shape on PaySim, via a
   shared business layer, with PaySim's balance columns deliberately excluded from the "safe"
   feature set pending further diagnostic work.

**Next steps before writing the manuscript:**
- Re-run end-to-end with `SAMPLE_N_IEEE = None` / `SAMPLE_N_PAYSIM = None` for the numbers that
  will actually appear in the paper (this notebook is written to be sample-size-agnostic).
- Sanity-check the velocity-loop runtime on the full IEEE-CIS table; if too slow, vectorize with
  `pandas.merge_asof` per entity or move to a rolling-window library.
- Decide on the final title / novelty claim **after** seeing the E2/E4 deltas — if behavioral
  enrichment doesn't move PR-AUC materially, that's still a valid (if less flattering) result to
  report honestly.
